# Formal evaluation — toxic-bert on val / test splits

Uses `split_indices.parquet` and `src/encoder.py` + `src/evaluate.py`.

**CLI equivalent:**
```bash
python scripts/make_splits.py
python scripts/run_formal_eval.py --quick
```

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

QUICK_N = 500  # cap per split; set None for full val/test
THRESHOLD = 0.5

_cwd = Path.cwd()
_bootstrap = _cwd.parent if _cwd.name == "notebooks" else _cwd
_src = str((_bootstrap / "src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

for _mod in ("paths", "device", "data", "splits", "encoder", "evaluate"):
    sys.modules.pop(_mod, None)

from encoder import ToxicBertEncoder
from evaluate import evaluate_multilabel_frame
from paths import (
    BENCHMARK_RESULTS_PATH,
    MODEL_ID,
    SPLIT_INDICES_PATH,
    TOXIC_LABELS,
)
from splits import sample_split_frame, split_counts

sns.set_theme(style="whitegrid")
print(f"Model: {MODEL_ID}")
print(f"Splits file: {SPLIT_INDICES_PATH} (exists={SPLIT_INDICES_PATH.exists()})")

## 1. Split sizes

In [ ]:
if not SPLIT_INDICES_PATH.exists():
    raise FileNotFoundError("Run: python scripts/make_splits.py")

display(split_counts())

## 2. Score val + test

In [ ]:
encoder = ToxicBertEncoder()
prob_cols = [f"enc_p_{l}" for l in TOXIC_LABELS]
pred_cols = [f"enc_pred_{l}" for l in TOXIC_LABELS]

results = {}
for split_name in ("val", "test"):
    df = sample_split_frame(split_name, n=QUICK_N)
    texts = df["comment_text"].astype(str).tolist()
    print(f"{split_name}: scoring {len(texts):,} comments ...")
    pred_df = encoder.predict_df(texts, threshold=THRESHOLD)
    gold = df[["comment_text"] + TOXIC_LABELS]
    results[split_name] = evaluate_multilabel_frame(
        gold, pred_df, prob_cols=prob_cols, pred_cols=pred_cols, threshold=THRESHOLD
    )
    s = results[split_name]["summary"]
    print(
        f"  macro_f1={s['macro_f1']:.3f} micro_f1={s['micro_f1']:.3f} "
        f"mean_pr_auc={s['mean_pr_auc']:.3f}"
    )

## 3. Per-label metrics

In [ ]:
for split_name, block in results.items():
    print(f"\n=== {split_name} (n={block['n']}) ===")
    display(block["per_label"].round(3))

## 4. Charts

In [ ]:
for split_name, block in results.items():
    per = block["per_label"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(data=per, x="label", y="f1", ax=axes[0], color="steelblue")
    axes[0].set_title(f"Per-label F1 — {split_name}")
    axes[0].tick_params(axis="x", rotation=30)
    sns.barplot(data=per, x="label", y="pr_auc", ax=axes[1], color="coral")
    axes[1].set_title(f"Per-label PR-AUC — {split_name}")
    axes[1].tick_params(axis="x", rotation=30)
    fig.tight_layout()
    plt.show()

## 5. Save benchmark JSON (optional)

In [ ]:
payload = {
    "model_id": MODEL_ID,
    "quick_n_per_split": QUICK_N,
    "threshold": THRESHOLD,
    "splits": {
        k: {
            "n": v["n"],
            "summary": v["summary"],
            "per_label": v["per_label"].round(4).to_dict(orient="records"),
        }
        for k, v in results.items()
    },
}
BENCHMARK_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(BENCHMARK_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
print(f"Wrote {BENCHMARK_RESULTS_PATH}")